# Region Analysis of Civilian Awards (Q1 – region part)
**Team 007 | An Analysis of India's Civilian and Military Honours**

How has the regional distribution of civilian award winners changed between the first 40 years (1954–1993) and the later period (1994–2021)?

**H0:** Region distribution has remained the same.

Steps: load data → map State/Country to region → split into two periods → region-wise CSVs per award → Plotly charts → chi-square test.

## 1. Setup and configuration

In [ ]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

INPUT_CSV = "../datasets/cleaned_datasets/national_awards_cleaned.csv"
OUT_DIR = "region_output"
# Data runs 1954-2021. "First 40 years" = 1954-1993, "next" = 1994-2021 (only 28 years of data,
# so shares (%) are used alongside counts). For the latest 40 years use PERIOD_2 = (1982, 2021).
PERIOD_1 = (1954, 1993)
PERIOD_2 = (1994, 2021)
AWARDS = ["Bharat Ratna", "Padma Vibhushan", "Padma Bhushan", "Padma Shri"]

## 2. State/Country → Region mapping

In [2]:
REGION_MAP = {
    # North
    "Delhi": "North", "Punjab": "North", "Haryana": "North", "Chandigarh": "North",
    "Himachal Pradesh": "North", "Jammu and Kashmir": "North", "Ladakh": "North",
    "Uttarakhand": "North", "Uttar Pradesh": "North", "Rajasthan": "North",
    # South
    "Tamil Nadu": "South", "Karnataka": "South", "Kerala": "South",
    "Telangana": "South", "Andhra Pradesh": "South", "Puducherry": "South",
    "Lakshadweep": "South",
    # East
    "West Bengal": "East", "Odisha": "East", "Bihar": "East", "Jharkhand": "East",
    "Andaman and Nicobar": "East",
    # West
    "Maharashtra": "West", "Gujarat": "West", "Goa": "West", "Daman and Diu": "West",
    # Central
    "Madhya Pradesh": "Central", "Chhattisgarh": "Central",
    # North-East
    "Assam": "North-East", "Manipur": "North-East", "Mizoram": "North-East",
    "Meghalaya": "North-East", "Sikkim": "North-East", "Nagaland": "North-East",
    "Arunachal Pradesh": "North-East", "Tripura": "North-East",
}
REGION_ORDER = ["North", "South", "East", "West", "Central", "North-East", "Foreign"]

def to_region(place: str) -> str:
    # anything not an Indian state/UT in the map is a foreign country
    return REGION_MAP.get(place, "Foreign")

def assign_period(year: int) -> str | None:
    if PERIOD_1[0] <= year <= PERIOD_1[1]:
        return f"{PERIOD_1[0]}-{PERIOD_1[1]}"
    if PERIOD_2[0] <= year <= PERIOD_2[1]:
        return f"{PERIOD_2[0]}-{PERIOD_2[1]}"
    return None

P1_LABEL = f"{PERIOD_1[0]}-{PERIOD_1[1]}"
P2_LABEL = f"{PERIOD_2[0]}-{PERIOD_2[1]}"

def slug(name: str) -> str:
    return name.lower().replace(" ", "_")

## 3. Load and prepare data

In [3]:
def load_data() -> pd.DataFrame:
    df = pd.read_csv(INPUT_CSV)
    df.columns = df.columns.str.strip()
    df["State/Country"] = df["State/Country"].astype(str).str.strip()
    df["Award"] = df["Award"].astype(str).str.strip()
    df["Region"] = df["State/Country"].map(to_region)
    df["Period"] = df["Year"].apply(assign_period)
    df = df.dropna(subset=["Period"])
    return df

df = load_data()
print(f"Rows used: {len(df)}  ({P1_LABEL}: {(df.Period == P1_LABEL).sum()}, {P2_LABEL}: {(df.Period == P2_LABEL).sum()})")
df.head()

Rows used: 4875  (1954-1993: 2347, 1994-2021: 2528)


,Name,Award,Field,State/Country,Year,Year of Birth,AgeAtAward,Region,Period
0,Dr. Sarvapalli Radhakrishnan,Bharat Ratna,Public Affairs,Tamil Nadu,1954,1888.0,66.0,South,1954-1993
1,Shri Chakravarti Rajagopalachari,Bharat Ratna,Public Affairs,Tamil Nadu,1954,1878.0,76.0,South,1954-1993
2,Shri Chandrasekhara Venkata Raman,Bharat Ratna,Science and Engineering,Tamil Nadu,1954,1888.0,66.0,South,1954-1993
3,Dr. Satyendra Nath Bose,Padma Vibhushan,Literature and Education,West Bengal,1954,1894.0,60.0,East,1954-1993
4,Dr. Zakir Husain,Padma Vibhushan,Public Affairs,Telangana,1954,1897.0,57.0,South,1954-1993


## 4. Region-wise summaries

In [4]:
def region_summary(sub: pd.DataFrame) -> pd.DataFrame:
    """Zone-level counts and % share for each period, sorted region-wise."""
    out = (sub.groupby(["Period", "Region"]).size().reset_index(name="Count"))
    # make sure every region/period pair exists (fill 0)
    full = pd.MultiIndex.from_product([[P1_LABEL, P2_LABEL], REGION_ORDER],
                                      names=["Period", "Region"])
    out = out.set_index(["Period", "Region"]).reindex(full, fill_value=0).reset_index()
    out["Share_%"] = (out["Count"] / out.groupby("Period")["Count"].transform("sum") * 100).round(2)
    out["Region"] = pd.Categorical(out["Region"], REGION_ORDER, ordered=True)
    out["Period"] = pd.Categorical(out["Period"], [P1_LABEL, P2_LABEL], ordered=True)
    return out.sort_values(["Period", "Region"]).reset_index(drop=True)

def state_summary(sub: pd.DataFrame) -> pd.DataFrame:
    """State-level counts, grouped by region, sorted by count within region."""
    out = (sub.groupby(["Period", "Region", "State/Country"]).size().reset_index(name="Count"))
    out["Share_%"] = (out["Count"] / out.groupby("Period")["Count"].transform("sum") * 100).round(2)
    out["Region"] = pd.Categorical(out["Region"], REGION_ORDER, ordered=True)
    out["Period"] = pd.Categorical(out["Period"], [P1_LABEL, P2_LABEL], ordered=True)
    return out.sort_values(["Period", "Region", "Count"],
                           ascending=[True, True, False]).reset_index(drop=True)

## 5. Plotly charts

In [5]:
def plot_region_share(reg: pd.DataFrame, award: str, path: str):
    fig = px.bar(reg, x="Region", y="Share_%", color="Period", barmode="group",
                 text="Count", category_orders={"Region": REGION_ORDER},
                 title=f"{award}: regional share of winners ({P1_LABEL} vs {P2_LABEL})",
                 labels={"Share_%": "Share of winners in period (%)"})
    fig.update_traces(textposition="outside")
    fig.write_html(path)
    return fig

def plot_top_states(st: pd.DataFrame, award: str, path: str, n: int = 10):
    top = st.groupby("State/Country")["Count"].sum().nlargest(n).index
    d = st[st["State/Country"].isin(top)]
    fig = px.bar(d, x="State/Country", y="Share_%", color="Period", barmode="group",
                 category_orders={"State/Country": list(top)},
                 title=f"{award}: top {n} states/countries ({P1_LABEL} vs {P2_LABEL})",
                 labels={"Share_%": "Share of winners in period (%)"})
    fig.write_html(path)
    return fig

def plot_overview(all_reg: pd.DataFrame, path: str):
    """One panel per award, zone share by period."""
    awards = list(all_reg["Award"].unique())
    fig = make_subplots(rows=2, cols=2, subplot_titles=awards)
    colors = {P1_LABEL: "#1f77b4", P2_LABEL: "#ff7f0e"}
    for i, award in enumerate(awards):
        r, c = divmod(i, 2)
        for p in (P1_LABEL, P2_LABEL):
            d = all_reg[(all_reg["Award"] == award) & (all_reg["Period"] == p)]
            fig.add_trace(go.Bar(x=d["Region"], y=d["Share_%"], name=p,
                                 marker_color=colors[p], legendgroup=p,
                                 showlegend=(i == 0)), row=r + 1, col=c + 1)
    fig.update_layout(barmode="group", height=750,
                      title="Regional share of civilian award winners by award")
    fig.update_yaxes(title_text="Share (%)")
    fig.write_html(path)
    return fig

def plot_heatmap(df: pd.DataFrame, path: str):
    """All Padma/Bharat Ratna winners: region x 5-year bin (counts as %)."""
    d = df.copy()
    d["Bin"] = (d["Year"] // 5 * 5).astype(str)
    piv = d.groupby(["Bin", "Region"]).size().unstack(fill_value=0).reindex(columns=REGION_ORDER, fill_value=0)
    piv = piv.div(piv.sum(axis=1), axis=0) * 100
    fig = px.imshow(piv.T.round(1), text_auto=True, aspect="auto", color_continuous_scale="Blues",
                    labels=dict(x="5-year bin (start)", y="Region", color="% of winners"),
                    title="Regional share of all civilian winners over time (5-year bins)")
    fig.write_html(path)
    return fig

## 6. Chi-square test
H0: region distribution is the same in both periods. Unreliable where most expected counts are < 5 (Bharat Ratna, Padma Vibhushan).

In [6]:
def chi_square(sub: pd.DataFrame, award: str):
    from scipy.stats import chi2_contingency
    tab = pd.crosstab(sub["Period"], sub["Region"])
    tab = tab.loc[:, tab.sum() > 0]
    chi2, p, dof, exp = chi2_contingency(tab)
    low = (exp < 5).sum() / exp.size
    note = "" if low <= 0.2 else "  (WARNING: many expected counts < 5, result unreliable)"
    print(f"{award:16s} chi2={chi2:8.2f}  dof={dof}  p={p:.4g}{note}")
    return {"Award": award, "chi2": round(chi2, 3), "dof": dof, "p_value": p,
            "reliable": low <= 0.2}

## 7. Per-award CSVs and charts

In [7]:
os.makedirs(OUT_DIR, exist_ok=True)
all_reg, tests = [], []
for award in AWARDS:
    sub = df[df["Award"] == award]
    if sub.empty:
        continue
    reg, st = region_summary(sub), state_summary(sub)
    reg.to_csv(f"{OUT_DIR}/{slug(award)}_region_summary.csv", index=False)
    st.to_csv(f"{OUT_DIR}/{slug(award)}_state_summary.csv", index=False)
    print(f"\n=== {award} ===")
    display(reg)
    plot_region_share(reg, award, f"{OUT_DIR}/{slug(award)}_region_share.html").show()
    plot_top_states(st, award, f"{OUT_DIR}/{slug(award)}_top_states.html").show()
    all_reg.append(reg.assign(Award=award))
    tests.append(chi_square(sub, award))


=== Bharat Ratna ===


,Period,Region,Count,Share_%
0,1954-1993,North,7,24.14
1,1954-1993,South,7,24.14
2,1954-1993,East,6,20.69
3,1954-1993,West,7,24.14
4,1954-1993,Central,0,0.00
5,1954-1993,North-East,0,0.00
6,1954-1993,Foreign,2,6.90
7,1994-2021,North,6,31.58
8,1994-2021,South,3,15.79
9,1994-2021,East,1,5.26


Bharat Ratna     chi2=    5.75  dof=5  p=0.3315  (WARNING: many expected counts < 5, result unreliable)

=== Padma Vibhushan ===


,Period,Region,Count,Share_%
0,1954-1993,North,46,32.62
1,1954-1993,South,36,25.53
2,1954-1993,East,18,12.77
3,1954-1993,West,31,21.99
4,1954-1993,Central,4,2.84
5,1954-1993,North-East,2,1.42
6,1954-1993,Foreign,4,2.84
7,1994-2021,North,64,35.56
8,1994-2021,South,43,23.89
9,1994-2021,East,10,5.56


Padma Vibhushan  chi2=   13.56  dof=6  p=0.03494  (WARNING: many expected counts < 5, result unreliable)

=== Padma Bhushan ===


,Period,Region,Count,Share_%
0,1954-1993,North,228,31.93
1,1954-1993,South,167,23.39
2,1954-1993,East,100,14.01
3,1954-1993,West,164,22.97
4,1954-1993,Central,16,2.24
5,1954-1993,North-East,9,1.26
6,1954-1993,Foreign,30,4.20
7,1994-2021,North,192,33.86
8,1994-2021,South,144,25.40
9,1994-2021,East,26,4.59


Padma Bhushan    chi2=   59.83  dof=6  p=4.862e-11

=== Padma Shri ===


,Period,Region,Count,Share_%
0,1954-1993,North,545,37.25
1,1954-1993,South,297,20.30
2,1954-1993,East,176,12.03
3,1954-1993,West,316,21.60
4,1954-1993,Central,28,1.91
5,1954-1993,North-East,62,4.24
6,1954-1993,Foreign,39,2.67
7,1994-2021,North,545,30.93
8,1994-2021,South,422,23.95
9,1994-2021,East,164,9.31


Padma Shri       chi2=   89.65  dof=6  p=3.59e-17


## 8. Overview, heatmap and test results

In [8]:
plot_overview(pd.concat(all_reg), f"{OUT_DIR}/overview_all_awards.html").show()
plot_heatmap(df, f"{OUT_DIR}/region_heatmap_over_time.html").show()
results = pd.DataFrame(tests)
results.to_csv(f"{OUT_DIR}/chi_square_results.csv", index=False)
results

,Award,chi2,dof,p_value,reliable
0,Bharat Ratna,5.748,5,3.315352e-01,False
1,Padma Vibhushan,13.561,6,3.494487e-02,False
2,Padma Bhushan,59.835,6,4.862302e-11,True
3,Padma Shri,89.645,6,3.589913e-17,True
